In [1]:
import re
import os

In [2]:
print(os.getcwd())

d:\MSc Artificial Intelligence\COMP5012


In [3]:
file_path = "data/Modules (data for JAN assessment).txt"

In [4]:
def normalise_module_code(code: str) -> str:
    code = code.strip().upper()
    code = code.replace("M0D", "MOD")

    import re
    match = re.match(r"MOD(\d+)$", code)
    if match:
        return f"MOD{match.group(1).zfill(3)}"

    digits = re.findall(r"\d+", code)
    if digits:
        return f"MOD{digits[0].zfill(3)}"

    return code


def parse_module_line(line: str) -> dict:
    parts = line.strip().split("|")

    module_code = normalise_module_code(parts[0])
    staff_name = parts[1].strip()
    num_labs = int(parts[2].strip())

    conflicts = [
        normalise_module_code(c)
        for c in parts[3].split(",")
        if c.strip()
    ]

    return {
        "module_id": module_code,
        "staff": staff_name,
        "num_labs": num_labs,
        "conflicts": conflicts
    }


def load_modules(file_path):
    modules = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                modules.append(parse_module_line(line))

    return modules

In [5]:
modules = load_modules(file_path)

print(f"Loaded {len(modules)} modules")
modules[:3]

Loaded 17 modules


[{'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'num_labs': 2,
  'conflicts': ['MOD002',
   'MOD003',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD013']},
 {'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'num_labs': 2,
  'conflicts': ['MOD001',
   'MOD003',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD013']},
 {'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'num_labs': 2,
  'conflicts': ['MOD001',
   'MOD002',
   'MOD004',
   'MOD005',
   'MOD006',
   'MOD007',
   'MOD008',
   'MOD009',
   'MOD010',
   'MOD011',
   'MOD012',
   'MOD013']}]

In [6]:
def build_events(modules):
    """
    Convert module data into a flat list of teaching events.
    
    Each module creates:
    - 1 lecture event
    - num_labs lab events
    """
    events = []

    for module in modules:
        module_id = module["module_id"]
        staff = module["staff"]

        # Add lecture event
        events.append({
            "event_id": f"{module_id}_LEC",
            "module_id": module_id,
            "staff": staff,
            "event_type": "lecture"
        })

        # Add lab events
        for lab_num in range(1, module["num_labs"] + 1):
            events.append({
                "event_id": f"{module_id}_LAB{lab_num}",
                "module_id": module_id,
                "staff": staff,
                "event_type": "lab"
            })

    return events

In [7]:
events = build_events(modules)

print(f"Total number of events: {len(events)}")
events[:10]

Total number of events: 48


[{'event_id': 'MOD001_LEC',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lecture'},
 {'event_id': 'MOD001_LAB1',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD001_LAB2',
  'module_id': 'MOD001',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD002_LEC',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lecture'},
 {'event_id': 'MOD002_LAB1',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lab'},
 {'event_id': 'MOD002_LAB2',
  'module_id': 'MOD002',
  'staff': 'Dominykas Cleary',
  'event_type': 'lab'},
 {'event_id': 'MOD003_LEC',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lecture'},
 {'event_id': 'MOD003_LAB1',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_type': 'lab'},
 {'event_id': 'MOD003_LAB2',
  'module_id': 'MOD003',
  'staff': 'Zacharias Karstensen',
  'event_t

In [8]:
for event in events:
    print(event)

{'event_id': 'MOD001_LEC', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture'}
{'event_id': 'MOD001_LAB1', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD001_LAB2', 'module_id': 'MOD001', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD002_LEC', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lecture'}
{'event_id': 'MOD002_LAB1', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab'}
{'event_id': 'MOD002_LAB2', 'module_id': 'MOD002', 'staff': 'Dominykas Cleary', 'event_type': 'lab'}
{'event_id': 'MOD003_LEC', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lecture'}
{'event_id': 'MOD003_LAB1', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD003_LAB2', 'module_id': 'MOD003', 'staff': 'Zacharias Karstensen', 'event_type': 'lab'}
{'event_id': 'MOD004_LEC', 'module_id': 'MOD004', 'staff':

In [ ]:
num_lectures = sum(1 for event in events if event["event_type"] == "lecture")
num_labs = sum(1 for event in events if event["event_type"] == "lab")

print("Number of lectures:", num_lectures)
print("Number of labs:", num_labs)
print("Total events:", len(events))

Number of lectures: 17
Number of labs: 31
Total events: 48
